# Import annotations

Import a local CATMA JSON export and attach its traces and category annotations to the Letter objects. The reviewed-document directory identifies the remaining reviewed negative examples.

In [ ]:
from shared import paths
from IPython.display import display


## Merge annotation fragments

In [ ]:
from shared.classes import Letter, IntertextualTrace, Annotation
import json
import pickle

paths.prepare_directories()


def merge_and_assign_annotations(input_file, output_file):
    """Merge CATMA annotation fragments and attach them to traces."""
    with open(input_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
    merged_documents = []
    errors = []
    category_properties = {
        'Editorial and Publishing Processes': ['uncertain', 'temporal_perspective', 'target_novel', 'subcategory'],
        'Elements of Fictional Craft': ['target_novel', 'subcategory', 'uncertain', 'type_of_reference', 'temporal_perspective'],
        'Financial and Legal Aspects': ['target_novel', 'subcategory', 'temporal_perspective', 'uncertain'],
        'Gratitude and Acknowledgement': ['uncertain', 'subcategory', 'temporal_perspective', 'target_novel'],
        'Impact and Reception': ['subcategory', 'temporal_perspective', 'target_novel', 'uncertain'],
        'Inspirations and Influences': ['target_novel', 'temporal_perspective', 'uncertain', 'subcategory'],
        'Marketing and Promotion': ['uncertain', 'temporal_perspective', 'subcategory', 'target_novel'],
        'Quality and Evaluation': ['uncertain', 'subcategory', 'temporal_perspective', 'target_novel'],
        'Writing Process and Creative Insight': ['target_novel', 'temporal_perspective', 'subcategory', 'uncertain'],
        'intertextual_trace': ['markers', 'uncertain'],
    }
    for document in data.get('exportDocuments', []):
        intertextual_traces_by_id = {}
        other_annotations_by_id = {}
        document_title = document.get('sourceDocument', {}).get('title', 'Unknown Document')
        for annotation in document.get('annotations', []):
            if annotation.get('tagName') == 'intertextual_trace':
                trace_id = annotation['id']
                if trace_id not in intertextual_traces_by_id:
                    properties = annotation['properties']
                    uncertain = next(
                        (prop['values'][0] for prop in properties if prop['name'] == 'uncertain' and prop['values']),
                        'n',
                    )
                    intertextual_traces_by_id[trace_id] = {
                        'id': trace_id,
                        'startOffset': annotation['startOffset'],
                        'endOffset': annotation['endOffset'],
                        'phrase': annotation['phrase'],
                        'properties': {
                            'uncertain': uncertain,
                            'markers': next(
                                (prop['values'] for prop in properties if prop['name'] == 'markers'),
                                [],
                            ),
                        },
                        'annotations': [],
                    }
                else:
                    intertextual_traces_by_id[trace_id]['startOffset'] = min(
                        intertextual_traces_by_id[trace_id]['startOffset'],
                        annotation['startOffset'],
                    )
                    intertextual_traces_by_id[trace_id]['endOffset'] = max(intertextual_traces_by_id[trace_id]['endOffset'], annotation['endOffset'])
                    intertextual_traces_by_id[trace_id]['phrase'] += annotation['phrase']
        for annotation in document.get('annotations', []):
            if annotation.get('tagName') != 'intertextual_trace':
                annotation_id = annotation['id']
                category = annotation.get('tagName', '')
                required_properties = category_properties.get(category, [])
                try:
                    properties = {}
                    for prop_name in required_properties:
                        value = next(
                            (prop['values'][0] for prop in annotation['properties'] if prop['name'] == prop_name and prop['values']),
                            'n' if prop_name == 'uncertain' else None,
                        )
                        if value is None:
                            errors.append(
                                {
                                    'document': document_title,
                                    'annotation_id': annotation_id,
                                    'missing_property': prop_name,
                                    'annotation': annotation,
                                },
                            )
                        properties[prop_name] = value
                    if annotation_id not in other_annotations_by_id:
                        other_annotations_by_id[annotation_id] = {
                            'id': annotation_id,
                            'startOffset': annotation['startOffset'],
                            'endOffset': annotation['endOffset'],
                            'phrase': annotation['phrase'],
                            'category': category,
                            **properties,
                        }
                    else:
                        other_annotations_by_id[annotation_id]['startOffset'] = min(
                            other_annotations_by_id[annotation_id]['startOffset'],
                            annotation['startOffset'],
                        )
                        other_annotations_by_id[annotation_id]['endOffset'] = max(
                            other_annotations_by_id[annotation_id]['endOffset'],
                            annotation['endOffset'],
                        )
                        other_annotations_by_id[annotation_id]['phrase'] += annotation['phrase']
                except (IndexError, KeyError, TypeError) as e:
                    errors.append(
                        {
                            'document': document_title,
                            'annotation_id': annotation_id,
                            'error': str(e),
                            'annotation': annotation,
                        },
                    )
        for trace in intertextual_traces_by_id.values():
            trace['merged_annotations'] = []
            for annotation in other_annotations_by_id.values():
                if (
                    trace['startOffset'] <= annotation['startOffset']
                    and trace['endOffset'] >= annotation['endOffset']
                ):
                    trace['merged_annotations'].append(annotation)
            trace['merged_annotations'] = sorted(trace['merged_annotations'], key=lambda x: x['startOffset'])
        intertextual_traces = [
            {
                'intertextual_trace': {
                    'id': value['id'],
                    'startOffset': value['startOffset'],
                    'endOffset': value['endOffset'],
                    'phrase': value['phrase'],
                    'properties': value['properties'],
                    'merged_annotations': value['merged_annotations'],
                },
            }
             for value in intertextual_traces_by_id.values()
        ]
        merged_documents.append(
            {'document_title': document_title, 'intertextual_traces': intertextual_traces},
        )
    merged_documents = sorted(merged_documents, key=lambda x: x['document_title'])
    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(merged_documents, file, indent=4, ensure_ascii=False)
    if errors:
        with open(paths.ANNOTATION_DIR / 'errors_in_annotations.json', 'w', encoding='utf-8') as error_file:
            json.dump(errors, error_file, indent=4, ensure_ascii=False)
input_file = str(paths.CATMA_JSON)
output_file = str(paths.ANNOTATIONS_JSON)
merge_and_assign_annotations(input_file, output_file)


## Attach annotations

In [ ]:
import pickle
import json


def load_pickle_file(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)


def load_json_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)


def update_letters_with_annotations(letters, annotations):
    """Attach traces by phrase match, falling back to the document ID."""
    updated_letters = []
    for old_letter in letters:
        new_letter = Letter(
            id=old_letter.id,
            anno_text=old_letter.anno_text,
            day=getattr(old_letter, 'day', None),
            month=getattr(old_letter, 'month', None),
            year=getattr(old_letter, 'year', None),
            recipient=getattr(old_letter, 'recipient', None),
            description=getattr(old_letter, 'description', None),
            text=getattr(old_letter, 'text', None),
        )
        updated_letters.append(new_letter)
    phrase_matches = 0
    fallback_matches = 0
    unmatched = 0
    for document in annotations:
        doc_id = document.get('document_title', '[unknown ID]')
        for raw_trace in document.get('intertextual_traces', []):
            trace_data = raw_trace['intertextual_trace']
            phrase = trace_data['phrase'].strip()
            matched_letters = [l for l in updated_letters if phrase in l.anno_text]
            if len(matched_letters) == 1:
                letter = matched_letters[0]
            elif len(matched_letters) != 1:
                letter = next((l for l in updated_letters if l.id == doc_id), None)
                if letter:
                    if len(matched_letters) > 1:
                        print(f' Multiple phrase matches; falling back to ID: {doc_id}')
                    else:
                        print(f' No phrase match; falling back to ID: {doc_id}')
                    fallback_matches += 1
                else:
                    print(f' No letter found for fallback ID: {doc_id}')
                    print('Text: [LETTER TEXT]')
                    unmatched += 1
                    continue
            letter.reviewed = True
            trace = IntertextualTrace(
                id=trace_data['id'],
                start_offset=trace_data['startOffset'],
                end_offset=trace_data['endOffset'],
                phrase=trace_data['phrase'],
                markers=trace_data['properties'].get('markers', []),
                uncertain=trace_data['properties'].get('uncertain', 'n'),
            )
            for raw_annotation in trace_data.get('merged_annotations', []):
                annotation = Annotation(
                    start_offset=raw_annotation['startOffset'],
                    end_offset=raw_annotation['endOffset'],
                    phrase=raw_annotation['phrase'],
                    category=raw_annotation['category'],
                    subcategory=raw_annotation.get('subcategory'),
                    target_novel=raw_annotation.get('target_novel'),
                    temporal_perspective=raw_annotation.get('temporal_perspective'),
                    uncertain=raw_annotation.get('uncertain', 'n'),
                    type_of_reference=raw_annotation.get('type_of_reference'),
                )
                trace.add_annotation(annotation)
            letter.add_intertextual_trace(trace)
            phrase_matches += 1
    print(f'\n Assigned by phrase or document ID: {phrase_matches}')
    print(f' Used document_title fallback: {fallback_matches}')
    print(f' Unassigned: {unmatched}')
    return updated_letters


def save_pickle_file(file_path, data):
    with open(file_path, 'wb') as file:
        pickle.dump(data, file)
pickle_file = str(paths.LETTERS_PICKLE)
json_file = str(paths.ANNOTATIONS_JSON)
output_pickle_file = str(paths.CORPUS_PICKLE)
letters = load_pickle_file(pickle_file)
annotations = load_json_file(json_file)
updated_letters = update_letters_with_annotations(letters, annotations)
save_pickle_file(output_pickle_file, updated_letters)
for letter in updated_letters[:5]:
    print(letter)


## Reviewed negative examples

In [ ]:
import os
import pickle
from shared.utils import load_pickle

output_pickle_file = str(paths.CORPUS_PICKLE)
text_folder = str(paths.REVIEWED_DOCUMENTS)
letters = load_pickle()
id_to_letter = {letter.id: letter for letter in letters}
txt_ids = [filename[:-4] for filename in os.listdir(text_folder) if filename.endswith('.txt')]
updated = 0
for bid in txt_ids:
    letter = id_to_letter.get(bid)
    if letter and (not getattr(letter, 'reviewed', False)):
        letter.reviewed = True
        letter.is_negative_example = True
        updated += 1
with open(output_pickle_file, 'wb') as f:
    pickle.dump(letters, f)
print(f' {updated} letters were marked as reviewed negative examples.')
print(f' Saved to: {output_pickle_file}')
